# Quick UMAP — 그때그때 찍기

한 줄 함수만 사용. 캐시된 좌표를 즉시 로드해서 plt로 찍는다.
기본 데이터셋은 **ptb-xl + zzu-pecg** 만.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from scripts.umap_view import (
    quick, quick_age_na,
    get_coords, get_emb, get_labels, get_ages, is_normal_mask,
)
import numpy as np
import matplotlib.pyplot as plt

## 1) 한 줄로 데이터셋별 색

In [ ]:
quick('CPC')                                       # default = ['ptbxl', 'zzu']
quick('ECG-Founder')
quick('ST-MEM')

## 2) **나이대별 × Normal/Abnormal** — 핵심 보기 모드

각 나이 bin이 별도 axes. 해당 bin만 **초록=Normal / 빨강=Abnormal** 로 칠하고, 다른 나이대는 회색.
Normal 매핑: ptbxl→`NORM`, zzu→`Normal_ECG | Otherwise_normal_ECG`.

In [ ]:
quick_age_na()   # default bins = (0,18,40,60,80,200)

In [ ]:
# 여러 모델을 한 figure에 (각 행이 한 모델)
quick_age_na(
    age_bins=[0, 3, 6, 12, 18, 30, 40, 50, 60, 70, 80, 200],
)

In [ ]:
# 다른 bin 분할 — 자유롭게
quick_age_na(['CPC', 'ECG-Founder'], age_bins=[0, 30, 50, 70, 200])

## 3) 두 프리셋 나란히 (L2 ablation)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
quick('ECG-Founder', preset='orig',         ax=axes[0], show=False)
quick('ECG-Founder', preset='euclideanL2', ax=axes[1], show=False)
axes[0].set_title('orig (euclidean, L2 OFF)')
axes[1].set_title('euclideanL2 (L2 ON)')
plt.tight_layout(); plt.show()

## 4) 좌표만 받아서 자유롭게 plt

In [ ]:
coords, sizes = get_coords('ECG-Founder', ['ptbxl', 'zzu'])
print(coords.shape, sizes)

# 예: ptbxl만 표시
n_ptbxl = sizes['ptbxl']
plt.figure(figsize=(7, 6))
plt.scatter(coords[:n_ptbxl, 0], coords[:n_ptbxl, 1], s=2, alpha=0.4)
plt.title('ECG-Founder · ptbxl only'); plt.xticks([]); plt.yticks([])
plt.show()

## 5) 임베딩 / 라벨 / 나이 직접 접근

In [ ]:
emb = get_emb('CPC', 'ptbxl');           print('emb:', emb.shape)
lbl, cols = get_labels('ptbxl');          print('labels:', lbl.shape, cols)
ages = get_ages('ptbxl');                 print('ages:', ages.shape)
norm = is_normal_mask('ptbxl', len(emb)); print('normal frac:', norm.mean())

## 6) **진단(ICD-10)별 클러스터링 + 성인/소아 비교** — 핵심 baseline

비교 모델 4개 (CPC, ECG-FM, ECG-Founder, ECG-JEPA)에 대해 PTB-XL + ZZU의 진단 라벨을
ICD-10 prefix로 통일해 같은 진단을 cross-dataset으로 묶어 색칠.

각 모델 row마다 **3 cols**: `combined / adult ≥18 / pediatric <18`.
각 cell에서 해당 그룹만 진단색, 다른 그룹/미매핑은 회색.
오른쪽엔 combined 그룹의 코드별 silhouette / kNN-BACC.

매핑 데이터:
- PTBXL: `ptbxl_all_paper_labels.csv` (약자) → ICD prefix
- ZZU: `zzu_bench_labels.csv` (`is_*`) → ICD prefix (사용자 매핑)
- 미매핑/없음 = 회색

> PTBXL은 성인 21,410 / 소아 426, ZZU는 모두 소아 (12,327)
> → 자연스러운 demographic robustness 비교 setup

In [ ]:
from scripts.umap_view import quick_dx, dx_compatibility_table
import pandas as pd
pd.set_option('display.width', 160)

# 진단 분포 + 호환성 표 (PTBXL + ZZU)
compat = dx_compatibility_table()
print(compat.to_string(index=False))
print(f"\n호환 진단 (양쪽 모두 sample 있음): {compat.compatible.sum()} / 전체 {len(compat)}")

In [ ]:
# 4개 모델 × ptbxl+zzu, 상위 8개 진단 코드, 성인 ≥18 / 소아 <18 분리 (default)
fig, metrics = quick_dx(top_n=8)

In [ ]:
# 특정 진단 코드만 — 임상 의미 있는 부정맥 (AFib, RBBB, LBBB, S.Tachy, PVC)
quick_dx(include_codes=['I48', 'I45.1', 'I44.7', 'R00.0', 'I49.3'])

## 7) Subgroup gap 분석 — adult vs pediatric robustness

`metrics`는 `{model: {그룹: {ICD코드: {sil, bacc, n_pos}}}}` 구조.
각 모델·진단별 **adult vs pediatric BACC gap** 과 **worst-group BACC** 를 표로.

- gap이 작을수록 demographic-robust
- worst-group BACC가 높을수록 minority subgroup 성능 ↑
- 이 두 지표가 우리 모델 (추후 비교)의 핵심 이점

In [ ]:
# 직전 quick_dx() 호출의 metrics 활용 — 진단별 adult/pediatric BACC 비교
import pandas as pd

adult_key = next(k for k in next(iter(metrics.values())) if 'adult' in k)
ped_key   = next(k for k in next(iter(metrics.values())) if 'pediatric' in k)

rows = []
for model, by_grp in metrics.items():
    for code, mm_a in by_grp.get(adult_key, {}).items():
        mm_p = by_grp.get(ped_key, {}).get(code, {})
        bacc_a = mm_a.get('bacc')
        bacc_p = mm_p.get('bacc')
        if bacc_a is None or bacc_p is None:
            continue
        if not (np.isfinite(bacc_a) and np.isfinite(bacc_p)):
            continue
        rows.append({
            'model': model,
            'code': code,
            'adult_bacc': round(bacc_a, 3),
            'pedi_bacc':  round(bacc_p, 3),
            'gap':       round(abs(bacc_a - bacc_p), 3),
            'worst':     round(min(bacc_a, bacc_p), 3),
            'n_adult':   mm_a.get('n_pos', 0),
            'n_pedi':    mm_p.get('n_pos', 0),
        })

df = pd.DataFrame(rows)
# 모델별 평균 gap & worst (진단 평균)
summary = df.groupby('model')[['gap', 'worst']].mean().round(3)
summary = summary.reindex(['CPC', 'ECG-FM', 'ECG-Founder', 'ECG-JEPA'])
print('=== 모델별 평균 (낮은 gap + 높은 worst가 좋음) ===')
print(summary)

print('\n=== 진단별 상세 (gap 큰 순) ===')
print(df.sort_values(['model','gap'], ascending=[True, False]).to_string(index=False))